In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_online_retail.csv")

C:\Users\HP\AppData\Local\Temp\ipykernel_41048\1168573445.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/cleaned_online_retail.csv")


In [4]:
order_products = (
    df[["InvoiceNo", "StockCode"]]
    .drop_duplicates()
)

In [5]:
from itertools import combinations
from collections import Counter

pair_counts = Counter()

for products in order_products.groupby("InvoiceNo")["StockCode"]:
    product_list = list(products[1])
    
    for pair in combinations(sorted(product_list), 2):
        pair_counts[pair] += 1

In [6]:
pairs_df = pd.DataFrame(
    pair_counts.items(),
    columns=["Product_Pair", "Order_Count"]
)

pairs_df = pairs_df.sort_values(
    "Order_Count",
    ascending=False
)

In [7]:
pairs_df.head(20)

,Product_Pair,Order_Count
12352,"(22386, 85099B)",825
136734,"(22697, 22699)",767
6392,"(21931, 85099B)",724
17331,"(22411, 85099B)",680
6884,"(20725, 22383)",655
6853,"(20725, 20727)",641
274,"(22726, 22727)",639
915793,"(22697, 22698)",632
2182,"(20725, 22384)",606
1070004,"(22698, 22699)",598


In [8]:
total_orders = df["InvoiceNo"].nunique()

pairs_df["support"] = (
    pairs_df["Order_Count"] / total_orders
)

In [9]:
product_frequency = (
    order_products.groupby("StockCode")["InvoiceNo"]
    .nunique()
)

In [10]:
top_pairs = pairs_df.head(10)

print(top_pairs)

            Product_Pair  Order_Count   support
12352    (22386, 85099B)          825  0.041329
136734    (22697, 22699)          767  0.038423
6392     (21931, 85099B)          724  0.036269
17331    (22411, 85099B)          680  0.034065
6884      (20725, 22383)          655  0.032812
6853      (20725, 20727)          641  0.032111
274       (22726, 22727)          639  0.032011
915793    (22697, 22698)          632  0.031660
2182      (20725, 22384)          606  0.030358
1070004   (22698, 22699)          598  0.029957


In [11]:
pairs_df.to_csv(
    "../data/processed/product_pairs.csv",
    index=False
)